# GT Keyword Extraction and Processing Workflow (Herald)
This notebook analyzes the GT keyword extraction and tag processing workflow for the herald dataset. It includes configuration, keyword processing, tag extraction, aggregation, and summary of tag ratings.

In [1]:
# Configuration for Herald Analysis

dataset_name = "herald"
dataset_language = "english"
base_url = f"https://cs.uef.fi/~himat/WebRank/dataset_12/dataset_12/{dataset_name}"
num_pages = 100

from nltk.stem.snowball import SnowballStemmer
import nltk
from nltk.corpus import stopwords

stemmer = SnowballStemmer(dataset_language)
nltk.download('stopwords')
nltk_english_stopwords = set(stopwords.words(dataset_language))

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/muditha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
import re

def process_keywords(keywords):
    """Apply cleaning, stemming, and stopword filtering to a list of keywords. Returns a processed list of useful words."""
    processed = []
    for word in keywords:
        cleaned_word = re.sub(r'[^a-zA-Z]', '', word)
        if not cleaned_word:
            continue
        stemmed_word = stemmer.stem(cleaned_word.lower())
        if stemmed_word not in nltk_english_stopwords:
            processed.append(stemmed_word)
    return processed

In [3]:
from bs4 import BeautifulSoup

def extract_and_process_keywords_from_tag(html_text, tag_name):
    """Extracts keywords from the content inside the specified tag in the given HTML text, processes them using process_keywords, and returns a dictionary with keyword frequencies."""
    soup = BeautifulSoup(html_text, 'html.parser')
    elements = soup.find_all(tag_name)
    raw_keywords = []
    for elem in elements:
        text = elem.get_text(separator=' ')
        raw_keywords.extend(text.split())
    processed = process_keywords(raw_keywords)
    freq = {}
    for kw in processed:
        freq[kw] = freq.get(kw, 0) + 1
    return freq

In [4]:
def extract_available_tags(html_text):
    """
    Extracts and returns a list of unique HTML tag names present in the given html_text.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    tags = set([tag.name for tag in soup.find_all()])
    return sorted(tags)

In [5]:
from urllib.parse import urlparse, unquote
from collections import defaultdict
import urllib.request

tag_rating_sum = defaultdict(float)
tag_count = defaultdict(int)

for i in range(num_pages):
    gt_url = f"{base_url}/{i}/GT.txt"
    web_page_url = f"{base_url}/{i}"
    url_file_url = f"{base_url}/{i}/URL.txt"
    processed_keywords = []
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Copilot/1.0)"}
    gt_req = urllib.request.Request(gt_url, headers=headers)
    web_req = urllib.request.Request(web_page_url, headers=headers)
    url_req = urllib.request.Request(url_file_url, headers=headers)
    try:
        with urllib.request.urlopen(gt_req, timeout=5) as response:
            gt_text = response.read().decode("utf-8-sig").strip()
            gt_keywords = gt_text.split()
            processed_keywords = list(set(process_keywords(gt_keywords)))
    except Exception as e:
        print(f"Error processing GT for page {i}: {e}")
        continue
    # HTML tag ratings
    try:
        with urllib.request.urlopen(web_req, timeout=5) as web_response:
            html_text = web_response.read().decode("utf-8-sig").strip()
            tags = extract_available_tags(html_text)
            for tag in tags:
                result = extract_and_process_keywords_from_tag(html_text, tag)
                total_keywords = sum(result.values())
                matched_sum = sum(freq for kw, freq in result.items() if kw in processed_keywords)
                rating = matched_sum / total_keywords if total_keywords > 0 else 0
                tag_rating_sum[tag] += rating
                tag_count[tag] += 1
    except Exception as e:
        print(f"Error processing web page for page {i}: {e}")
        continue
    # URL rating as a pseudo-tag
    try:
        with urllib.request.urlopen(url_req, timeout=5) as url_response:
            real_url = url_response.read().decode("utf-8-sig").strip()
            parsed_url = urlparse(real_url)
            normalized_path = unquote(parsed_url.path.lower())
            url_tokens = re.findall(r"[a-zåäöA-ZÅÄÖ0-9]+", normalized_path)
            processed_url_keywords = process_keywords(url_tokens)
            total_url_keywords = len(processed_url_keywords)
            matched_url_keywords = sum(1 for kw in processed_url_keywords if kw in processed_keywords)
            rating = matched_url_keywords / total_url_keywords if total_url_keywords > 0 else 0
            tag_rating_sum['URL'] += rating
            tag_count['URL'] += 1
    except Exception as e:
        print(f"Error processing URL for page {i}: {e}")
        continue

# Calculate sum rating per tag and sort (including URL as a tag)
tag_sum_rating = {tag: tag_rating_sum[tag] for tag in tag_rating_sum}
sorted_tags = sorted(tag_sum_rating.items(), key=lambda x: x[1], reverse=True)

print("Tag and URL rating sum across all pages (sorted):")
for tag, rating_sum in sorted_tags:
    print(f"{tag}: {rating_sum:.4f}")

Tag and URL rating sum across all pages (sorted):
URL: 69.5116
h1: 54.3161
head: 39.1021
title: 39.1021
figcaption: 30.7214
figure: 30.7214
article: 14.9774
html: 6.6049
body: 6.0500
p: 5.0392
div: 4.9454
span: 4.7256
a: 2.4831
nav: 1.0833
li: 0.8121
ul: 0.8121
h3: 0.6528
small: 0.5000
strong: 0.4286
footer: 0.4091
thead: 0.2857
em: 0.2500
table: 0.0333
td: 0.0333
tr: 0.0333
br: 0.0000
form: 0.0000
h2: 0.0000
header: 0.0000
iframe: 0.0000
img: 0.0000
input: 0.0000
link: 0.0000
meta: 0.0000
style: 0.0000
sup: 0.0000
tbody: 0.0000
